In [ ]:
# Cell 1 — Setup
# Requires:
#   pip install -r requirements.txt
#   python -m spacy download en_core_web_sm
# Launch Jupyter from the repo root OR from the notebooks/ subdirectory.

import sys
from pathlib import Path

# Find repo root: start from cwd and walk up until grammar_parser/ is found.
REPO_ROOT = Path.cwd()
for _ in range(3):
    if (REPO_ROOT / 'grammar_parser').is_dir():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise RuntimeError(
        "Could not locate grammar_parser/ in cwd or its parents.\n"
        "Set REPO_ROOT manually, e.g.:\n"
        "  REPO_ROOT = Path(r'C:/Users/marta/Documents/PROJECT BARCELONA/GRAMMAR')"
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import spacy
from grammar_parser import Group1Parser, Group2Parser, Group3Parser, Group4Parser

nlp = spacy.load('en_core_web_sm')

parser1 = Group1Parser(nlp)   # LEXICAL_TRIGGER  — MODALITY, NEGATION, DISCOURSE MARKERS, CONJUNCTIONS, PREPOSITIONS
parser2 = Group2Parser(nlp)   # NOMINAL_POS      — PRONOUNS, DETERMINERS, NOUNS, ADJECTIVES, ADVERBS
parser3 = Group3Parser(nlp)   # VERBAL_MORPHOLOGY — PAST, PRESENT, FUTURE, PASSIVES, VERBS
parser4 = Group4Parser(nlp)   # SYNTACTIC_STRUCTURE — CLAUSES, REPORTED SPEECH, FOCUS, QUESTIONS

parsers = [parser1, parser2, parser3, parser4]

print(f"Repo root    : {REPO_ROOT}")
print(f"spaCy model  : {nlp.meta['name']} v{nlp.meta['version']}")
print(f"Group1Parser : {len(parser1.structures)} structures  (strategy1 — LEXICAL_TRIGGER)")
print(f"Group2Parser : {len(parser2.structures)} structures  (strategy2 — NOMINAL_POS)")
print(f"Group3Parser : {len(parser3.structures)} structures  (strategy3 — VERBAL_MORPHOLOGY)")
print(f"Group4Parser : {len(parser4.structures)} structures  (strategy4 — SYNTACTIC_STRUCTURE)")
print(f"─" * 52)
total = sum(len(p.structures) for p in parsers)
print(f"Total        : {total} structures across {len(parsers)} parsers")

In [2]:
# Cell 2 — Load lesson sentences
# Reads from validated-sentence-separation.txt (properly segmented sentences).
# Run separate_sentences.py first if the file doesn't exist yet.

LESSON_PATH = REPO_ROOT / 'processed_data' / 'sentences' / 'Student-1' / 'lesson-1' / 'validated-sentence-separation.txt'

def load_sentences(path: Path) -> list:
    with path.open(encoding='utf-8') as fh:
        lines = [line.strip() for line in fh]
    return [line for line in lines if line]

sentences = load_sentences(LESSON_PATH)

print(f"Loaded {len(sentences)} sentences from:")
print(f"  {LESSON_PATH}")
print()
print('First 5 sentences:')
for s in sentences[:5]:
    print(f'  {s!r}')

Loaded 302 sentences from:
  C:\Users\marta\Documents\PROJECT BARCELONA\GRAMMAR\processed_data\sentences\Student-1\lesson-1\validated-sentence-separation.txt

First 5 sentences:
  '\ufeffHello?'
  'Fine.'
  'Thanks.'
  'Marcos, how are you?'
  'Very good.'


In [3]:
# Cell 3 — Pipeline function
# run_pipeline accepts any list of parser objects that implement .parse(doc).
# Adding Group2Parser later: parsers = [parser1, parser2]

def run_pipeline(sentences, nlp, parsers):
    """
    Process all sentences through all parsers.

    Returns
    -------
    list of dict:
        sentence_index : int
        sentence_text  : str
        matches        : list[dict]  — aggregated results from all parsers,
                         each match includes start_char / end_char for highlighting
    """
    results = []
    docs = list(nlp.pipe(sentences))

    for idx, (sentence, doc) in enumerate(zip(sentences, docs)):
        all_matches = []
        for parser in parsers:
            for m in parser.parse(doc):
                # Attach character offsets so the display layer can highlight spans
                span = doc[m['start_token']:m['end_token']]
                m['start_char'] = span.start_char
                m['end_char']   = span.end_char
                all_matches.append(m)
        results.append({
            'sentence_index': idx,
            'sentence_text':  sentence,
            'matches':        all_matches,
        })

    return results

print('run_pipeline defined.')

run_pipeline defined.


In [4]:
# Cell 4 — Run the pipeline

pipeline_results = run_pipeline(sentences, nlp, parsers)

total_matches = sum(len(r['matches']) for r in pipeline_results)
sentences_with_hits = sum(1 for r in pipeline_results if r['matches'])

print('Pipeline complete.')
print(f'  Sentences processed  : {len(pipeline_results)}')
print(f'  Sentences with hits  : {sentences_with_hits}')
print(f'  Total match events   : {total_matches}')

Pipeline complete.
  Sentences processed  : 302
  Sentences with hits  : 129
  Total match events   : 2976


In [5]:
# Cell 5 — Visual display (HTML)
#
# For each sentence with matches:
#   • Full sentence text with matched spans highlighted (colour = Proficiency Dimension)
#   • Verification table: span | dimension | category | guideword | CEFR level
#
# Matches are capped at MAX_ROWS_PER_SPAN per category to keep the view readable.
# Use max_sentences to page through results.

from IPython.display import HTML, display as _display
from collections import defaultdict
import html as _html

# ── Proficiency Dimensions ────────────────────────────────────────────────────
DIMENSION_MAP = {
    "CLAUSES": "A",  "QUESTIONS": "A",  "REPORTED SPEECH": "A",
    "CONJUNCTIONS": "A",  "DISCOURSE MARKERS": "A",
    "PAST": "B",  "PRESENT": "B",  "FUTURE": "B",
    "PASSIVES": "B",  "VERBS": "B",  "NEGATION": "B",
    "PRONOUNS": "C",  "DETERMINERS": "C",  "NOUNS": "C",
    "ADJECTIVES": "C",  "ADVERBS": "C",  "PREPOSITIONS": "C",
    "MODALITY": "D",  "FOCUS": "D",
}
DIMENSION_LABELS = {
    "A": "Sentence Architecture",
    "B": "Tense & Aspect",
    "C": "Nominal Precision",
    "D": "Modal & Function",
}
DIM_BG   = {"A": "#DBEAFE", "B": "#FEF3C7", "C": "#D1FAE5", "D": "#EDE9FE", "?": "#F3F4F6"}
DIM_TEXT = {"A": "#1D4ED8", "B": "#92400E", "C": "#065F46", "D": "#5B21B6", "?": "#374151"}
DIM_MARK = {"A": "#93C5FD", "B": "#FCD34D", "C": "#6EE7B7", "D": "#C4B5FD", "?": "#D1D5DB"}

CEFR_BG  = {"A1": "#D1FAE5", "A2": "#6EE7B7", "B1": "#FEF3C7",
            "B2": "#FDE68A", "C1": "#FECACA", "C2": "#FCA5A5"}
CEFR_TXT = {"A1": "#065F46", "A2": "#064E3B", "B1": "#78350F",
            "B2": "#92400E", "C1": "#991B1B", "C2": "#7F1D1D"}


def _dim_badge(dim):
    bg = DIM_BG.get(dim, DIM_BG["?"])
    fg = DIM_TEXT.get(dim, DIM_TEXT["?"])
    label = DIMENSION_LABELS.get(dim, dim)
    return (f'<span style="background:{bg};color:{fg};padding:2px 8px;'
            f'border-radius:10px;font-size:11px;font-weight:600;white-space:nowrap;">'
            f'{dim} &middot; {label}</span>')


def _cefr_badge(level):
    bg = CEFR_BG.get(level, "#F3F4F6")
    fg = CEFR_TXT.get(level, "#374151")
    return (f'<span style="background:{bg};color:{fg};padding:2px 7px;'
            f'border-radius:8px;font-size:11px;font-weight:700;">{level}</span>')


def _highlight_sentence(text, char_spans):
    """
    char_spans: list of (start_char, end_char, dimension)
    Returns an HTML string with matched spans coloured by dimension.
    Overlapping spans are merged (first dimension wins).
    """
    # Merge overlapping / adjacent spans, keeping the first dimension
    merged = []
    for start, end, dim in sorted(char_spans):
        if merged and start < merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)   # extend, keep dim
        else:
            merged.append([start, end, dim])

    parts = []
    pos = 0
    for start, end, dim in merged:
        parts.append(_html.escape(text[pos:start]))
        mark_col = DIM_MARK.get(dim, DIM_MARK["?"])
        text_col = DIM_TEXT.get(dim, DIM_TEXT["?"])
        parts.append(
            f'<mark style="background:{mark_col};color:{text_col};'
            f'padding:1px 3px;border-radius:3px;font-weight:700;">'
            f'{_html.escape(text[start:end])}</mark>'
        )
        pos = end
    parts.append(_html.escape(text[pos:]))
    return ''.join(parts)


def display_results_html(pipeline_results, max_sentences=15, max_rows_per_cat=5):
    """
    Render pipeline results as styled HTML in Jupyter.

    Parameters
    ----------
    max_sentences    : max number of sentences (with matches) to show
    max_rows_per_cat : max guideword rows shown per (span, category) group
    """
    blocks = []
    shown  = 0

    for result in pipeline_results:
        matches = result['matches']
        if not matches:
            continue
        if shown >= max_sentences:
            remaining = sum(1 for r in pipeline_results if r['matches']) - shown
            blocks.append(
                f'<p style="color:#6B7280;font-style:italic;margin-top:8px;">'
                f'&#x2026; {remaining} more sentence(s) with matches not shown. '
                f'Increase <code>max_sentences</code> to see more.</p>'
            )
            break
        shown += 1

        idx           = result['sentence_index']
        sentence_text = result['sentence_text']

        # ── Build highlighted sentence ──────────────────────────────────────
        unique_spans = {}
        for m in matches:
            key = (m['start_char'], m['end_char'])
            if key not in unique_spans:
                unique_spans[key] = DIMENSION_MAP.get(m['category'], '?')

        highlighted = _highlight_sentence(
            sentence_text,
            [(s, e, d) for (s, e), d in unique_spans.items()]
        )

        # ── Build verification table ────────────────────────────────────────
        # Group: (start_char, end_char) → category → [matches]
        span_cat = defaultdict(lambda: defaultdict(list))
        for m in matches:
            span_cat[(m['start_char'], m['end_char'])][m['category']].append(m)

        rows = []
        for (s, e), cat_dict in sorted(span_cat.items()):
            span_raw = sentence_text[s:e]
            first_span_in_group = True
            for cat, cat_matches in sorted(cat_dict.items()):
                dim = DIMENSION_MAP.get(cat, '?')
                sorted_m = sorted(cat_matches, key=lambda x: x['lowest_level_numeric'])
                show     = sorted_m[:max_rows_per_cat]
                extra    = len(sorted_m) - max_rows_per_cat

                for i, m in enumerate(show):
                    # Span cell: only on first row of each span group
                    if first_span_in_group and i == 0:
                        span_cell = (
                            f'<code style="background:#F3F4F6;padding:2px 6px;'
                            f'border-radius:4px;font-size:12px;">'
                            f'&ldquo;{_html.escape(span_raw)}&rdquo;</code>'
                        )
                    else:
                        span_cell = ''

                    dim_cell = _dim_badge(dim) if i == 0 else ''
                    cat_cell = (
                        f'<span style="font-size:11px;font-weight:600;color:#374151;">'
                        f'{_html.escape(cat)}</span>'
                    ) if i == 0 else ''

                    rows.append(
                        f'<tr style="border-bottom:1px solid #F3F4F6;">'
                        f'<td style="padding:5px 8px;vertical-align:top;">{span_cell}</td>'
                        f'<td style="padding:5px 8px;vertical-align:top;">{dim_cell}</td>'
                        f'<td style="padding:5px 8px;vertical-align:top;">{cat_cell}</td>'
                        f'<td style="padding:5px 8px;font-size:12px;color:#1F2937;">'
                        f'{_html.escape(m["guideword"])}</td>'
                        f'<td style="padding:5px 8px;text-align:center;">'
                        f'{_cefr_badge(m["lowest_level"])}</td>'
                        f'</tr>'
                    )
                    first_span_in_group = False

                if extra > 0:
                    rows.append(
                        f'<tr><td colspan="5" style="padding:2px 8px 6px;font-size:11px;'
                        f'color:#6B7280;font-style:italic;">'
                        f'&nbsp;&nbsp;&nbsp;+ {extra} more {_html.escape(cat)} guideword(s)'
                        f'</td></tr>'
                    )

        blocks.append(f'''
        <div style="margin:14px 0;padding:14px 16px;border:1px solid #E5E7EB;
                    border-radius:8px;font-family:system-ui,-apple-system,sans-serif;
                    background:#FAFAFA;">
          <div style="font-size:11px;color:#9CA3AF;margin-bottom:6px;
                      letter-spacing:.04em;">SENTENCE {idx:03d}</div>
          <div style="font-size:14px;line-height:1.8;margin-bottom:12px;
                      padding:8px 12px;background:#FFFFFF;border-radius:6px;
                      border-left:3px solid #D1D5DB;">
            {highlighted}
          </div>
          <table style="width:100%;border-collapse:collapse;font-size:12px;">
            <thead>
              <tr style="background:#F9FAFB;border-bottom:2px solid #E5E7EB;">
                <th style="padding:6px 8px;text-align:left;color:#6B7280;
                           font-size:10px;text-transform:uppercase;letter-spacing:.06em;">Matched span</th>
                <th style="padding:6px 8px;text-align:left;color:#6B7280;
                           font-size:10px;text-transform:uppercase;letter-spacing:.06em;">Dimension</th>
                <th style="padding:6px 8px;text-align:left;color:#6B7280;
                           font-size:10px;text-transform:uppercase;letter-spacing:.06em;">Category</th>
                <th style="padding:6px 8px;text-align:left;color:#6B7280;
                           font-size:10px;text-transform:uppercase;letter-spacing:.06em;">Guideword</th>
                <th style="padding:6px 8px;text-align:center;color:#6B7280;
                           font-size:10px;text-transform:uppercase;letter-spacing:.06em;">Level</th>
              </tr>
            </thead>
            <tbody>{''.join(rows)}</tbody>
          </table>
        </div>''')

    # ── Dimension legend ──────────────────────────────────────────────────────
    legend_items = '&nbsp;&nbsp;'.join(_dim_badge(d) for d in 'ABCD')
    legend = (
        f'<div style="margin-bottom:12px;padding:10px 16px;background:#F9FAFB;'
        f'border-radius:8px;font-family:system-ui,sans-serif;'
        f'border:1px solid #E5E7EB;">'
        f'<span style="font-size:11px;font-weight:600;color:#374151;'
        f'margin-right:12px;">Proficiency Dimensions</span>'
        f'{legend_items}</div>'
    )

    _display(HTML(legend + '\n'.join(blocks)))


display_results_html(pipeline_results)

In [6]:
# Cell 6 — Summary by Proficiency Dimension + CEFR level

def display_summary_html(pipeline_results):
    from collections import Counter

    dim_counts   = Counter()
    level_counts = Counter()
    cat_counts   = Counter()

    sentences_hit = 0
    for result in pipeline_results:
        if result['matches']:
            sentences_hit += 1
        for m in result['matches']:
            dim = DIMENSION_MAP.get(m['category'], '?')
            dim_counts[dim]          += 1
            level_counts[m['lowest_level']] += 1
            cat_counts[m['category']]        += 1

    total = sum(dim_counts.values())
    if total == 0:
        _display(HTML('<p>No matches found.</p>'))
        return

    # ── CEFR bar chart ────────────────────────────────────────────────────────
    max_lv = max(level_counts.values())
    lv_rows = []
    for lv in ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']:
        n   = level_counts.get(lv, 0)
        pct = n / total * 100
        w   = n / max_lv * 240
        bg  = CEFR_BG.get(lv, '#E5E7EB')
        lv_rows.append(
            f'<tr>'
            f'<td style="padding:4px 10px 4px 0;vertical-align:middle;">{_cefr_badge(lv)}</td>'
            f'<td style="padding:4px 4px;vertical-align:middle;">'
            f'<div style="background:{bg};height:16px;width:{w:.0f}px;'
            f'border-radius:3px;display:inline-block;min-width:4px;"></div></td>'
            f'<td style="padding:4px 8px;font-size:12px;color:#374151;'
            f'white-space:nowrap;">{n:,} &nbsp;<span style="color:#9CA3AF;">({pct:.1f}%)</span></td>'
            f'</tr>'
        )

    # ── Dimension bar chart ───────────────────────────────────────────────────
    max_dm = max(dim_counts.values()) if dim_counts else 1
    dm_rows = []
    for dim in 'ABCD':
        n   = dim_counts.get(dim, 0)
        pct = n / total * 100
        w   = n / max_dm * 240
        mk  = DIM_MARK.get(dim, '#D1D5DB')
        dm_rows.append(
            f'<tr>'
            f'<td style="padding:4px 10px 4px 0;vertical-align:middle;">{_dim_badge(dim)}</td>'
            f'<td style="padding:4px 4px;vertical-align:middle;">'
            f'<div style="background:{mk};height:16px;width:{w:.0f}px;'
            f'border-radius:3px;display:inline-block;min-width:4px;"></div></td>'
            f'<td style="padding:4px 8px;font-size:12px;color:#374151;'
            f'white-space:nowrap;">{n:,} &nbsp;<span style="color:#9CA3AF;">({pct:.1f}%)</span></td>'
            f'</tr>'
        )

    # ── Category breakdown (top 10) ───────────────────────────────────────────
    cat_rows = []
    for cat, n in cat_counts.most_common(10):
        dim  = DIMENSION_MAP.get(cat, '?')
        pct  = n / total * 100
        mark = DIM_MARK.get(dim, '#D1D5DB')
        w    = n / cat_counts.most_common(1)[0][1] * 180
        cat_rows.append(
            f'<tr style="border-bottom:1px solid #F3F4F6;">'
            f'<td style="padding:4px 8px;font-size:12px;font-weight:600;color:#1F2937;">'
            f'{_html.escape(cat)}</td>'
            f'<td style="padding:4px 8px;">{_dim_badge(dim)}</td>'
            f'<td style="padding:4px 8px;">'
            f'<div style="background:{mark};height:14px;width:{w:.0f}px;'
            f'border-radius:3px;display:inline-block;min-width:4px;"></div></td>'
            f'<td style="padding:4px 8px;font-size:12px;color:#374151;'
            f'white-space:nowrap;">{n:,} &nbsp;<span style="color:#9CA3AF;">({pct:.1f}%)</span></td>'
            f'</tr>'
        )

    html = f'''
    <div style="font-family:system-ui,-apple-system,sans-serif;padding:16px 0;">
      <h3 style="color:#1F2937;margin:0 0 6px 0;font-size:16px;">Results Summary</h3>
      <p style="font-size:13px;color:#6B7280;margin:0 0 20px 0;">
        {len(pipeline_results)} sentences &nbsp;&middot;&nbsp;
        {sentences_hit} with matches &nbsp;&middot;&nbsp;
        {total:,} total match events
      </p>

      <div style="display:flex;gap:40px;flex-wrap:wrap;align-items:flex-start;">

        <div>
          <div style="font-size:11px;font-weight:600;color:#6B7280;
                      text-transform:uppercase;letter-spacing:.06em;
                      margin-bottom:8px;">CEFR Level</div>
          <table style="border-collapse:collapse;">{''.join(lv_rows)}</table>
        </div>

        <div>
          <div style="font-size:11px;font-weight:600;color:#6B7280;
                      text-transform:uppercase;letter-spacing:.06em;
                      margin-bottom:8px;">Proficiency Dimension</div>
          <table style="border-collapse:collapse;">{''.join(dm_rows)}</table>
        </div>

        <div>
          <div style="font-size:11px;font-weight:600;color:#6B7280;
                      text-transform:uppercase;letter-spacing:.06em;
                      margin-bottom:8px;">Top Categories</div>
          <table style="border-collapse:collapse;">
            <thead>
              <tr style="background:#F9FAFB;border-bottom:2px solid #E5E7EB;">
                <th style="padding:6px 8px;text-align:left;font-size:10px;color:#9CA3AF;">Category</th>
                <th style="padding:6px 8px;text-align:left;font-size:10px;color:#9CA3AF;">Dimension</th>
                <th colspan="2" style="padding:6px 8px;text-align:left;font-size:10px;color:#9CA3AF;">Count</th>
              </tr>
            </thead>
            <tbody>{''.join(cat_rows)}</tbody>
          </table>
        </div>

      </div>
    </div>
    '''
    _display(HTML(html))


display_summary_html(pipeline_results)

A1,,469 (15.8%)
A2,,732 (24.6%)
B1,,642 (21.6%)
B2,,599 (20.1%)
C1,,346 (11.6%)
C2,,188 (6.3%)
A · Sentence Architecture,,176 (5.9%)
B · Tense & Aspect,,511 (17.2%)
C · Nominal Precision,,981 (33.0%)
D · Modal & Function,,"1,308 (44.0%)"
Category,Dimension,Count
